In [2]:
import pandas as pd
import numpy as np
from nltk import sent_tokenize
from nltk import word_tokenize
from tqdm import tqdm
from Levenshtein import ratio
import os
import nltk
stop=nltk.corpus.stopwords.words('english')

In [8]:
df=pd.read_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_all.feather')
data=pd.read_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_contexts.feather')

In [16]:
df.shape

(53614, 6)

In [11]:
df['text'].iloc[2]

'IPWPJIIWP JvaajMiill II IIIHIRWWMPMMI ft V i Vf V t ki frl y iOUf mm MI Am Hi fflWMtSEMPEROR EULOGIZES DISORDER AftD feiil MAGISTRATES RAP Ink AAiniT Ar TUP ll ll rJflKIIUMIItMUA Ministers and Others Join in t Praiso of Evoning Publio Ledger for Expos lOTRY IS DENOUNCED Honest Courage of Expose Praised by Business Men To thu Editor ot Kutnlnp Public Ieiqrr Sir It nlwnyn was a known fnct thnt the mnrnltiR Pcmtc TKnonn utood for Juaticr tirnl rWitpmiMiem and It teem thnt Its ofNiirins the Evieino Pinuc Lepokr is fol lowing tlie footsteps of tlie fnther publication At n mcotlnc of the South Street Buitnesi Men Association held Thursday September 15 It wns Ilnnnlmoitslr revived thnt n vote of thanks be extended to your vnlun ble paper for It honest eonrnce and conviction In cxponltiK the Ku Klix Kln n ronglomerntlon of hoodlum and swindlers masquerading ns patriotic Amerlrans IIoplnB that you may carry on the food work of exposing the sorailed True Americans II M LEVY President S VRAM Vic

#### Sliding window of 3

In [4]:
candidate_article_index=[]
candidate_window_index=[]
candidate=[]
candidate_ratio=[]
for article_index, article in tqdm(df['text'].items(), total=len(df['text'])):
    tokens = [w for w in word_tokenize(str(article)) if w.strip()]  
    tokens = [t.lower() for t in tokens]
    windows = [
        ' '.join(tokens[i:i+3])   
        for i in range(len(tokens) - 2)
    ]
    for window in windows:
        ratio_value = ratio(window, 'ku klux klan')
        if (ratio_value >= 0.8) and (ratio_value < 1.0): #0.7 includes articles like "klux klan and" (0.72) or "klux klan in" (0.75)
            candidate_article_index.append(article_index)
            candidate_window_index.append(windows.index(window))
            candidate.append(window)
            candidate_ratio.append(ratio_value)

  0%|          | 0/53614 [00:00<?, ?it/s]

100%|██████████| 53614/53614 [00:24<00:00, 2194.33it/s]


In [17]:
candidate_df=pd.DataFrame({'article_index': candidate_article_index, 'window_index': candidate_window_index, 'window': candidate, 'ratio': candidate_ratio})

In [19]:
candidate_df[candidate_df['window']=='ku klux klnn']

,article_index,window_index,window,ratio
3,2,194,ku klux klnn,0.916667
226,142,165,ku klux klnn,0.916667
349,38,42,ku klux klnn,0.916667
432,22,180,ku klux klnn,0.916667
499,33,136,ku klux klnn,0.916667
521,28,125,ku klux klnn,0.916667
579,13,185,ku klux klnn,0.916667
585,70,167,ku klux klnn,0.916667
761,8,50,ku klux klnn,0.916667
786,144,45,ku klux klnn,0.916667


#### Find the context of the candidate three token

In [11]:
def extract_candidate_contexts(text, phrase, window):
    """
    Return list of context strings (joined tokens) for each match of phrase in text.
    Also returns the start token index for each match (useful for position).
    """
    if not phrase:
        return []

    tokens = word_tokenize(text.lower())
    phrase_tokens = word_tokenize(phrase.lower())
    windows = []
    n = len(tokens)
    m = len(phrase_tokens)

    # iterate only where phrase can fit
    for i in range(n - m + 1):
        if tokens[i:i + m] == phrase_tokens:
            start = max(0, i - window)
            end = min(n, i + m + window)   # use m (phrase length) not a fixed 3
            window_tokens = tokens[start:end]
            # you might want to show the phrase highlighted; here we just return the window text
            windows.append({
                'match_start_token_idx': i,
                'context_tokens': window_tokens,
                'context_text': " ".join(window_tokens)
            })
    return windows

In [29]:
results = []  # list of dicts

for article_index, article in tqdm(df[~df['text'].isna()]['text'].items(), total=len(df[~df['text'].isna()]['text'])):
    for candidate_phrase in pd.Series(candidate).unique():   # candidate assumed to be an iterable of phrases
        candidate_contexts = extract_candidate_contexts(article, candidate_phrase, window=5)
        if candidate_contexts:
            # append one result per found context to preserve alignment
            for ctx in candidate_contexts:
                results.append({
                    'article_index': article_index,
                    'candidate': candidate_phrase,
                    'match_start_token_idx': ctx['match_start_token_idx'],
                    'context_text': ctx['context_text'],
                    'context_tokens': ctx['context_tokens']
                })

100%|██████████| 52546/52546 [3:20:16<00:00,  4.37it/s]  


In [31]:
pd.DataFrame(results).to_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_contexts.feather')

#### Find exact "ku klux klan"

In [12]:
kkk_results=[]
for article_index, article in tqdm(df[~df['text'].isna()]['text'].items(), total=len(df[~df['text'].isna()]['text'])):
    candidate_contexts=extract_candidate_contexts(article, "ku klux klan", window=5)
    for ctx in candidate_contexts:
        kkk_results.append({
            "article_index": article_index,
            'candidate': "ku klux klan",
            'match_start_token_idx': ctx['match_start_token_idx'],
            'context_text': ctx['context_text'],
            'context_tokens': ctx['context_tokens']
        })
    

100%|██████████| 52546/52546 [00:21<00:00, 2462.75it/s]


In [18]:
pd.DataFrame(kkk_results).shape

(4463, 5)

In [15]:
pd.DataFrame(kkk_results).to_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_contexts_2.feather')

#### Find "klan" 

In [24]:
klan_results=[]
for article_index, article in tqdm(df[~df['text'].isna()]['text'].items(), total=len(df[~df['text'].isna()]['text'])):
    tokens = word_tokenize(article.lower())
    for token_idx, token in enumerate(tokens):
        if token == 'klan':
            context_text = " ".join(tokens[max(0, token_idx-5):token_idx+6])
            klan_results.append({
            "article_index": article_index,
            'candidate': "klan",
            'match_start_token_idx': token_idx,
            'context_text': context_text,
            'context_tokens': tokens[max(0, token_idx-5):token_idx+6]
            })  

100%|██████████| 52546/52546 [00:19<00:00, 2684.81it/s]


In [25]:
pd.DataFrame(klan_results)

,article_index,candidate,match_start_token_idx,context_text,context_tokens
0,0,klan,43,knights of the ku klux klan north star klan no 2,"[knights, of, the, ku, klux, klan, north, star..."
1,0,klan,46,ku klux klan north star klan no 2 of minneapol...,"[ku, klux, klan, north, star, klan, no, 2, of,..."
2,0,klan,64,defence of the ku klux klan reprint from liter...,"[defence, of, the, ku, klux, klan, reprint, fr..."
3,0,klan,75,20 1923 the ku klux klan has been charged with...,"[20, 1923, the, ku, klux, klan, has, been, cha..."
4,0,klan,118,stance louisiana members of the klan were char...,"[stance, louisiana, members, of, the, klan, we..."
...,...,...,...,...,...
10398,156,klan,26,large traugott sussmans and former klan chiefs...,"[large, traugott, sussmans, and, former, klan,..."
10399,156,klan,179,w lee smith lawyer former klan grand dragon ro...,"[w, lee, smith, lawyer, former, klan, grand, d..."
10400,156,klan,186,dragon robert f mcnay former klan titian dr fr...,"[dragon, robert, f, mcnay, former, klan, titia..."
10401,157,klan,17,number 24 hooded ku klux klan organhed at west...,"[number, 24, hooded, ku, klux, klan, organhed,..."


In [26]:
pd.DataFrame(klan_results).to_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_contexts_3.feather')

In [27]:
pd.DataFrame(klan_results)

,article_index,candidate,match_start_token_idx,context_text,context_tokens
0,0,klan,43,knights of the ku klux klan north star klan no 2,"[knights, of, the, ku, klux, klan, north, star..."
1,0,klan,46,ku klux klan north star klan no 2 of minneapol...,"[ku, klux, klan, north, star, klan, no, 2, of,..."
2,0,klan,64,defence of the ku klux klan reprint from liter...,"[defence, of, the, ku, klux, klan, reprint, fr..."
3,0,klan,75,20 1923 the ku klux klan has been charged with...,"[20, 1923, the, ku, klux, klan, has, been, cha..."
4,0,klan,118,stance louisiana members of the klan were char...,"[stance, louisiana, members, of, the, klan, we..."
...,...,...,...,...,...
10398,156,klan,26,large traugott sussmans and former klan chiefs...,"[large, traugott, sussmans, and, former, klan,..."
10399,156,klan,179,w lee smith lawyer former klan grand dragon ro...,"[w, lee, smith, lawyer, former, klan, grand, d..."
10400,156,klan,186,dragon robert f mcnay former klan titian dr fr...,"[dragon, robert, f, mcnay, former, klan, titia..."
10401,157,klan,17,number 24 hooded ku klux klan organhed at west...,"[number, 24, hooded, ku, klux, klan, organhed,..."
